# **BilSTM**

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


BiLSTM — это двунаправленная LSTM: она смотрит на последовательность и слева-направо, и справа-налево одновременно.

По сути это “LSTM с двумя мозгами”:

первый читает ряд как обычно: от прошлого к будущему;

второй — наоборот: от будущего к прошлому;

их состояния склеиваются и подаются дальше

In [ ]:
df = pd.read_csv("Brent.csv")
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,AO_saucer_down,EntrySignal,EntryReason,Fractal_Up_conf,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct,AddOn_Ready,AddOn_Triggered
0,2015-10-26 10:00:00,48.05,48.12,47.89,48.09,48.28815,48.15758,48.06107,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
1,2015-10-26 11:00:00,48.10,48.36,48.00,48.30,48.26098,48.12601,48.05886,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
2,2015-10-26 12:00:00,48.30,48.35,48.18,48.30,48.23552,48.11588,48.05209,0,0,...,0,-1,three_color,0,0,NaN,NaN,NaN,0,0
3,2015-10-26 13:00:00,48.30,48.34,48.05,48.09,48.21971,48.10765,48.04267,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
4,2015-10-26 14:00:00,48.11,48.28,47.97,48.07,48.19550,48.09732,48.07014,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36239,2025-10-21 23:00:00,61.55,61.73,61.55,61.65,61.17612,61.14427,61.29423,0,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36240,2025-10-22 09:00:00,62.27,62.66,62.26,62.46,61.16565,61.21749,61.31439,1,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36241,2025-10-22 10:00:00,62.45,62.49,62.19,62.35,61.13752,61.23530,61.35151,0,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36242,2025-10-22 11:00:00,62.35,62.55,62.25,62.36,61.14002,61.25526,61.40921,0,0,...,0,0,NaN,1,0,61.52,1.0,0.3,0,0


In [ ]:
scale_cols = [
    "Open", "High", "Low", "Close",
    "Alligator_Jaw", "Alligator_Teeth", "Alligator_Lips",
    "AO",
    "AddOn_Anchor_Level", "AddOn_Size_Pct"
]

scaler = RobustScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])


In [ ]:
columns = [
    "AddOn_Anchor_Level",
    "AddOn_Anchor_IsUp",
    "AddOn_Size_Pct"
]

df = df.dropna(subset=columns).reset_index(drop=True)

In [ ]:
df = pd.read_csv("Brent.csv")

H = 20  # горизонт удержания сделки

def add_goodtrade_target(df, horizon=20):
    df = df.copy()

    # будущая цена
    df["Close_fwd"] = df["Close"].shift(-horizon)

    # доходность по направлению сигнала
    ret_long  = (df["Close_fwd"] - df["Close"]) / df["Close"]
    ret_short = (df["Close"] - df["Close_fwd"]) / df["Close"]

    df["ret_H"] = np.where(
        df["EntrySignal"] > 0, ret_long,
        np.where(df["EntrySignal"] < 0, ret_short, 0.0)
    )

    # таргет
    df["GoodTrade"] = ((df["EntrySignal"] != 0) & (df["ret_H"] > 0)).astype(int)

    # убираем хвост, где нет Close_fwd
    df = df.iloc[:-horizon].reset_index(drop=True)
    return df

df = add_goodtrade_target(df, horizon=H)

print(df[["Close", "Close_fwd", "ret_H", "EntrySignal", "GoodTrade"]].head())



   Close  Close_fwd     ret_H  EntrySignal  GoodTrade
0  48.09      46.70  0.000000            0          0
1  48.30      46.77  0.000000            0          0
2  48.30      46.81  0.030849           -1          1
3  48.09      46.58  0.000000            0          0
4  48.07      46.77  0.000000            0          0


In [ ]:
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,Fractal_Up_conf,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct,AddOn_Ready,AddOn_Triggered,Close_fwd,ret_H,GoodTrade
0,2015-10-26 10:00:00,48.05,48.12,47.89,48.09,48.28815,48.15758,48.06107,0,0,...,0,0,NaN,NaN,NaN,0,0,46.70,0.000000,0
1,2015-10-26 11:00:00,48.10,48.36,48.00,48.30,48.26098,48.12601,48.05886,0,0,...,0,0,NaN,NaN,NaN,0,0,46.77,0.000000,0
2,2015-10-26 12:00:00,48.30,48.35,48.18,48.30,48.23552,48.11588,48.05209,0,0,...,0,0,NaN,NaN,NaN,0,0,46.81,0.030849,1
3,2015-10-26 13:00:00,48.30,48.34,48.05,48.09,48.21971,48.10765,48.04267,0,0,...,0,0,NaN,NaN,NaN,0,0,46.58,0.000000,0
4,2015-10-26 14:00:00,48.11,48.28,47.97,48.07,48.19550,48.09732,48.07014,0,0,...,0,0,NaN,NaN,NaN,0,0,46.77,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36219,2025-10-20 18:00:00,60.73,61.04,60.51,60.60,61.31318,61.20603,61.06995,0,0,...,0,1,63.53,0.0,0.3,0,0,61.65,0.000000,0
36220,2025-10-20 19:00:00,60.68,60.79,60.63,60.73,61.29409,61.16590,60.96996,0,0,...,0,0,63.53,0.0,0.3,0,0,62.46,0.000000,0
36221,2025-10-20 20:00:00,60.73,61.06,60.66,60.98,61.28646,61.13203,60.92697,0,0,...,0,0,63.53,0.0,0.3,0,0,62.35,0.000000,0
36222,2025-10-20 21:00:00,60.98,61.14,60.92,60.96,61.27366,61.06178,60.89658,1,0,...,0,0,63.53,0.0,0.3,0,0,62.36,0.000000,0


In [ ]:
# Берём только числовые колонки
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Что точно НЕ идёт в признаки
drop_feature_cols = ["Close_fwd", "ret_H", "GoodTrade"]

feature_cols = [c for c in numeric_cols if c not in drop_feature_cols]
print("Фичи для BiLSTM:", feature_cols)

# Сплит по времени на train / test
split_bar = int(len(df) * 0.8)
train_df = df.iloc[:split_bar].copy()
test_df  = df.iloc[split_bar:].copy()

# Масштабируем ТОЛЬКО по train
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

print("Train bars:", len(train_df), "Test bars:", len(test_df))
print("Доля GoodTrade=1 в train:", train_df["GoodTrade"].mean())
print("Доля GoodTrade=1 в test :", test_df["GoodTrade"].mean())


Фичи для BiLSTM: ['Open', 'High', 'Low', 'Close', 'Alligator_Jaw', 'Alligator_Teeth', 'Alligator_Lips', 'Fractal_Up', 'Fractal_Down', 'AO', 'Color AO', 'Alligator_Bullish', 'Alligator_Bearish', 'AlligatorStart_Long', 'AlligatorStart_Short', 'AO_sign', 'AO_zero_up', 'AO_zero_down', 'AO_three_green', 'AO_three_red', 'AO_saucer_up', 'AO_saucer_down', 'EntrySignal', 'Fractal_Up_conf', 'Fractal_Down_conf', 'AddOn_Anchor_Level', 'AddOn_Anchor_IsUp', 'AddOn_Size_Pct', 'AddOn_Ready', 'AddOn_Triggered']
Train bars: 28979 Test bars: 7245
Доля GoodTrade=1 в train: 0.02425894613340695
Доля GoodTrade=1 в test : 0.023740510697032435


In [ ]:
SEQ_LEN = 50  # длина окна истории, можно изменить

def make_sequences(df_part, feature_cols, seq_len=50):
    data = df_part[feature_cols].values
    targets = df_part["GoodTrade"].values
    signals = df_part["EntrySignal"].values

    X_list, y_list = [], []

    for i in range(seq_len - 1, len(df_part)):
        # нас интересуют только бары, где есть сигнал
        if signals[i] == 0:
            continue

        X_seq = data[i - seq_len + 1 : i + 1, :]  # [seq_len, num_features]
        y_val = targets[i]

        X_list.append(X_seq)
        y_list.append(y_val)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    return X, y

X_train, y_train = make_sequences(train_df, feature_cols, seq_len=SEQ_LEN)
X_test, y_test   = make_sequences(test_df,  feature_cols, seq_len=SEQ_LEN)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("Доля GoodTrade=1 в train seq:", y_train.mean())
print("Доля GoodTrade=1 в test seq :", y_test.mean())


X_train: (28930, 50, 30) X_test: (7196, 50, 30)
Доля GoodTrade=1 в train seq: 0.024230902177670238
Доля GoodTrade=1 в test seq : 0.02376320177876598


In [ ]:
num_features = X_train.shape[2]

model = Sequential([
    Bidirectional(
        LSTM(64, return_sequences=False),
        input_shape=(SEQ_LEN, num_features)
    ),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")  # бинарная классификация
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 40s 78ms/step - accuracy: 0.9573 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 2/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 26s 58ms/step - accuracy: 0.9741 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 3/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 27s 59ms/step - accuracy: 0.9779 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 4/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 40s 58ms/step - accuracy: 0.9748 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 5/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 41s 58ms/step - accuracy: 0.9759 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 6/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 41s 58ms/step - accuracy: 0.9759 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 7/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 26s 58ms/step - accuracy: 0.9757 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 8/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 27s 60ms/step - accuracy: 0.9760 - loss: nan - val_accuracy: 0.9762 - val_loss: nan


In [ ]:
y_proba = model.predict(X_test).ravel()
y_pred = (y_proba >= 0.5).astype(int)

print("BiLSTM AUC:", roc_auc_score(y_test, y_proba))
print("\nОтчёт по классификации (BiLSTM):")
print(classification_report(y_test, y_pred))
print("Матрица ошибок (BiLSTM):")
print(confusion_matrix(y_test, y_pred))


225/225 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step


ValueError: Input contains NaN.